# LGM50LT 21700 cells
- Capacity: 4.8 Ah
- Min Voltage 2.5 V
- Max Voltage 4.2 V 

In [ ]:
import pybamm
import numpy as np
import pickle
import pandas as pd
from scipy.integrate import cumulative_trapezoid
import matplotlib.pyplot as plt

# pybamm.set_logging_level("INFO")
pybamm.set_logging_level("WARNING")

data_DIR = "../data/"
exp_DIR = data_DIR + "experiment/"
out_DIR = data_DIR + "output/"
fig_DIR = "../figs/"
%matplotlib widget

In [ ]:
# load existing LGM50 data set
parameter_values = pybamm.ParameterValues("Chen2020")

In [ ]:
# load experimental data
file_path = exp_DIR + "LGM50L21700_CELL003_RPT_1_RT_0p0PSI_20251124_R0_RPT_21700M50.xlsx"
sheet_name = "record"
df = pd.read_excel(file_path, sheet_name = sheet_name)


In [ ]:
# model
model = pybamm.lithium_ion.DFN()

## Charge

In [ ]:
df1 = df[(df["Step Index"] == 13)]
V_data = df1["Voltage(V)"]
I_data = -df1["Current(A)"]
Q_data = df1["Capacity(Ah)"]
time = df1["Time"]
# Convert to seconds
t = pd.to_timedelta(time).dt.total_seconds().to_numpy()
# Compute differences
dt = np.diff(t, prepend=t[0])
# Enforce monotonic increase:
# If dt <= 0, force dt = 0
dt = np.maximum(dt, 0)
# Build increasing time array
time_axis = np.cumsum(dt)
t_data = time_axis
CC_data = (cumulative_trapezoid(np.abs(I_data), t_data))/3600

In [ ]:
crate = "0.25A"
period = "1s"
experiment = pybamm.Experiment(
    [
        f"Charge at {crate} until 4.2V",
        # "Hold at 4.2 V until C/100",
        # f"Discharge at {crate} until 3V",
    ],
    period = period,
)

In [ ]:
sim = pybamm.Simulation(
    model,
    experiment=experiment,
    parameter_values=parameter_values,
)
solution = sim.solve(initial_soc=0)

In [ ]:
t_sim = solution["Time [s]"].entries
V_sim = solution["Terminal voltage [V]"].entries
I_sim = solution["Current [A]"].entries

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(10,4))
ax1 = ax.flat[0]
ax1.plot(t_data,V_data,'b')
ax1.plot(t_sim,V_sim,'r--')
ax1.set_xlabel("Time [s]")
ax1.set_ylabel("Voltage [V]")
ax2 = ax.flat[1]
ax2.plot(t_data,I_data,'b')
ax2.plot(t_sim,I_sim,'r--')
ax2.set_xlabel("Time [s]")
ax2.set_ylabel("Current [A]")
ax2.legend(["data","sim"])
fig.suptitle("Fresh Cell Simulation")
fig.tight_layout()
fig.savefig(fig_DIR+"lgm50_cell003_fresh_charge.png",dpi=600)

## Discharge

In [ ]:
df1 = df[(df["Step Index"] == 15)]
V_data = df1["Voltage(V)"]
I_data = -df1["Current(A)"]
Q_data = df1["Capacity(Ah)"]
time = df1["Time"]
# Convert to seconds
t = pd.to_timedelta(time).dt.total_seconds().to_numpy()
# Compute differences
dt = np.diff(t, prepend=t[0])
# Enforce monotonic increase:
# If dt <= 0, force dt = 0
dt = np.maximum(dt, 0)
# Build increasing time array
time_axis = np.cumsum(dt)
t_data = time_axis
CC_data = (cumulative_trapezoid(np.abs(I_data), t_data))/3600

In [ ]:
crate = "0.25A"
period = "1s"
experiment = pybamm.Experiment(
    [
        # f"Charge at {crate} until 4.2V",
        # "Hold at 4.2 V until C/100",
        f"Discharge at {crate} until 2.5V",
    ],
    period = period,
)

In [ ]:
sim = pybamm.Simulation(
    model,
    experiment=experiment,
    parameter_values=parameter_values,
)
solution = sim.solve(initial_soc=1)

In [ ]:
t_sim = solution["Time [s]"].entries
V_sim = solution["Terminal voltage [V]"].entries
I_sim = solution["Current [A]"].entries

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(10,4))
ax1 = ax.flat[0]
ax1.plot(t_data,V_data,'b')
ax1.plot(t_sim,V_sim,'r--')
ax1.set_xlabel("Time [s]")
ax1.set_ylabel("Voltage [V]")
ax2 = ax.flat[1]
ax2.plot(t_data,I_data,'b')
ax2.plot(t_sim,I_sim,'r--')
ax2.set_xlabel("Time [s]")
ax2.set_ylabel("Current [A]")
ax2.legend(["data","sim"])
fig.suptitle("Fresh Cell Simulation")
fig.tight_layout()
fig.savefig(fig_DIR+"lgm50_cell003_fresh_discharge.png",dpi=600)

## HPPC

In [ ]:
df1 = df[(df["Cycle Index"] > 0) & (df["Cycle Index"] < 9) & (df["Step Index"] > 1)]
V_data = df1["Voltage(V)"].to_numpy()
I_data = -df1["Current(A)"].to_numpy()
Q_data = df1["Capacity(Ah)"].to_numpy()
time = df1["Time"]
# Convert to seconds
t = pd.to_timedelta(time).dt.total_seconds().to_numpy()
# Compute differences
dt = np.diff(t, prepend=t[0])
# Enforce monotonic increase:
# If dt <= 0, force dt = 0
dt = np.maximum(dt, 0)
# Build increasing time array
time_axis = np.cumsum(dt)
t_data = time_axis
CC_data = (cumulative_trapezoid(np.abs(I_data), t_data))/3600

In [ ]:
# _, first_indices = np.unique(t_data, return_index=True)
# first_indices_sorted = np.sort(first_indices)
# t_data_u = t_data[first_indices_sorted]
# I_data_u = I_data[first_indices_sorted]
# V_data_u = V_data[first_indices_sorted]

In [ ]:
fig, ax = plt.subplots()
ax.plot(t_data,I_data)
ax.set_xlim([0,7000])

In [ ]:
period = "1s"
experiment = pybamm.Experiment(
    [
        "Rest for 2 min",
        "Discharge at 0.5A for 60 min",
        "Rest for 15 min",
        "Charge at 5A for 5 sec",
        "Rest for 5 min",
        "Discharge at 5A for 10 sec",
        "Rest for 5 min",
    ]*9,
    period = period,
)

In [ ]:
# create interpolant - must be a function of *dimensional* time
current_interpolant = pybamm.Interpolant(t_data, I_data, pybamm.t)
# set drive cycle
parameter_values["Current function [A]"] = current_interpolant

sim = pybamm.Simulation(
    model,
    parameter_values=parameter_values,
)
solution = sim.solve(initial_soc=0.999)

In [ ]:
t_sim = solution["Time [s]"].entries
V_sim = solution["Terminal voltage [V]"].entries
I_sim = solution["Current [A]"].entries

In [ ]:
idx_c = np.where((I_sim[:-1] < -4.9) & (I_sim[:-1] > -5.1) & (I_sim[1:] == 0) )[0] 

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(10,4))
ax1 = ax.flat[0]
ax1.plot(t_data,V_data,'b')
ax1.plot(t_sim,V_sim,'r--')
ax1.set_xlabel("Time [s]")
ax1.set_ylabel("Voltage [V]")
ax2 = ax.flat[1]
ax2.plot(t_data,I_data,'b')
ax2.plot(t_sim,I_sim,'r--')
ax2.set_xlabel("Time [s]")
ax2.set_ylabel("Current [A]")
ax2.legend(["data","sim"])
fig.suptitle("Fresh Cell Simulation")
fig.tight_layout()
fig.savefig(fig_DIR+"lgm50_cell003_fresh_hppc.png",dpi=600)

In [ ]:
fig, axi = plt.subplots(3,3,figsize=(5*3,4*3))
a = 0
for i in range(len(idx_c)):
    ax = axi.flat[i]
    b = a + 2*60+60*60+15*60+5+5*60+10+5*60
    ax.plot(t_data,V_data,'b')
    ax.plot(t_sim,V_sim,'r--')
    ax.set_xlim([a,b+240])
    ax.set_ylim([V_sim[idx_c[i]]-0.3,V_sim[idx_c[i]]+0.01])
    a = b
    ax.set_title(f"SOC = {(0.9-i*0.1):0.1f}")
    ax.set_xlabel("Time [s]")
    ax.set_ylabel("Voltage [V]")
ax = axi.flat[0]
ax.legend(["data","sim"])
fig.tight_layout()
fig.savefig(fig_DIR+"lgm50_cell003_fresh_hppc_spilt_1.png",dpi=600)


In [ ]:
fig, axi = plt.subplots(3,3,figsize=(5*3,4*3))
a = 2*60+60*60+15*60
for i in range(len(idx_c)):
    ax = axi.flat[i]
    b = a + 5+5*60+10+5*60
    ax.plot(t_data,V_data,'b')
    ax.plot(t_sim,V_sim,'r--')
    ax.set_xlim([a-120,b+120])
    ax.set_ylim([V_sim[idx_c[i]]-0.3,V_sim[idx_c[i]]+0.01])
    a = b + 2*60+60*60+15*60
    ax.set_title(f"SOC = {(0.9-i*0.1):0.1f}")
    ax.set_xlabel("Time [s]")
    ax.set_ylabel("Voltage [V]")
ax = axi.flat[0]
ax.legend(["data","sim"])
fig.tight_layout()
fig.savefig(fig_DIR+"lgm50_cell003_fresh_hppc_spilt_2.png",dpi=600)


In [ ]:
fig, axi = plt.subplots(3,3,figsize=(5*3,4*3))
a = 2*60+60*60+15*60
for i in range(len(idx_c)):
    ax = axi.flat[i]
    ax.plot(t_data,V_data,'b')
    ax.plot(t_sim,V_sim,'r--')
    ax.set_xlim([a-30,a+30])
    ax.set_ylim([V_sim[idx_c[i]]-0.2,V_sim[idx_c[i]]+0.01])
    a+= 2*60+60*60+15*60+5+5*60+10+5*60
    ax.set_title(f"SOC = {(0.9-i*0.1):0.1f}")
    ax.set_xlabel("Time [s]")
    ax.set_ylabel("Voltage [V]")
ax = axi.flat[0]
ax.legend(["data","sim"])
fig.tight_layout()
fig.savefig(fig_DIR+"lgm50_cell003_fresh_hppc_spilt_3.png",dpi=600)


In [ ]:
fig, axi = plt.subplots(3,3,figsize=(5*3,4*3))
a = 2*60+60*60+15*60+5+5*60
for i in range(len(idx_c)):
    ax = axi.flat[i]
    ax.plot(t_data,V_data,'b')
    ax.plot(t_sim,V_sim,'r--')
    ax.set_xlim([a-30,a+30])
    ax.set_ylim([V_sim[idx_c[i]]-0.3,V_sim[idx_c[i]]-0.1])
    a+= 2*60+60*60+15*60+5+5*60+10+5*60
    ax.set_title(f"SOC = {(0.9-i*0.1):0.1f}")
    ax.set_xlabel("Time [s]")
    ax.set_ylabel("Voltage [V]")
ax = axi.flat[0]
ax.legend(["data","sim"])
fig.tight_layout()
fig.savefig(fig_DIR+"lgm50_cell003_fresh_hppc_spilt_4.png",dpi=600)
